# Two-qubit entanglement

*quantum · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/quantum/two_qubit_entanglement)

Two qubits stored as a $(2, 2)$ tensor $\psi_{ab}$. Gates are rank-4 tensors applied with one einsum: a Hadamard on qubit 1 followed by a CNOT turns $\lvert 00\rangle$ into a Bell state. The singular value decomposition of $\psi_{ab}$ is the Schmidt decomposition, giving the entanglement entropy directly. The pair then evolves under an Ising Hamiltonian; partial traces give each qubit's reduced density matrix, whose Bloch vector shrinks inside the sphere whenever the qubits are entangled.

**Vector form**

$$|\psi\rangle = \sum_{ab}\psi_{ab}\,|a\rangle|b\rangle,\quad |\Phi^+\rangle = \mathrm{CNOT}\,(H\otimes I)\,|00\rangle,\quad \hat H = J\,\sigma_z\otimes\sigma_z + h\,(\sigma_x\otimes I + I\otimes\sigma_x),\quad S = -\mathrm{Tr}\,\rho_1\ln\rho_1$$

**Index (tensor) form**

$$\psi_{ab} \leftarrow G_{abcd}\,\psi_{cd},\qquad \psi_{ab} = U_{ak}\,s_k\,V^*_{bk},\quad S = -s_k^2\ln s_k^2,\qquad \rho^{(1)}_{ac} = \psi_{ab}\,\psi^*_{cb},\quad r_k = \sigma_{kca}\,\rho^{(1)}_{ac}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\psi_{ab}$ | `basis_state(0, 0)` | (2, 2) complex | amplitude for qubit 1 in $\lvert a\rangle$ and qubit 2 in $\lvert b\rangle$ |
| $G_{abcd}$ | `einsum('ac,bd->abcd', Hd, I)  /  cnot()` | (2, 2, 2, 2) complex | two-qubit gate as a rank-4 tensor: output indices $a\,b$, input indices $c\,d$ |
| $G_{abcd}\psi_{cd}$ | `einsum('abcd,cd->ab', G, psi)` | (2, 2) | $c$ and $d$ summed: apply the gate |
| $s_k$ | `torch.linalg.svdvals(psi)` | (2,) | Schmidt coefficients; one nonzero value ⇔ product state |
| $\rho^{(1)}_{ac}$ | `einsum('ab,cb->ac', psi, psi.conj())` | (2, 2) | partial trace over qubit 2 ($b$ summed) |
| $H_{abcd}$ | `J*einsum('ac,bd->abcd', Z, Z) + h*(...)` | (2, 2, 2, 2) | Ising Hamiltonian built from tensor products, reshaped to (4, 4) for eigh |
| $r_k$ | `einsum('kca,ac->k', sigma, rho)` | (3,) | Bloch vector $\operatorname{Tr}(\rho\,\sigma_k)$ of one qubit; $\lvert\mathbf r\rvert < 1$ means mixed |

Units $\hbar = 1$. The tensor product of single-qubit operators $A\otimes B$ is the rank-4 tensor $A_{ac}B_{bd}$ (torch.kron gives the same numbers as a $4\times 4$ matrix with row index $2a + b$). Because $\psi_{ab}$ is a matrix, its SVD writes the state as $\sum_k s_k\lvert u_k\rangle\lvert v_k\rangle$: the Schmidt decomposition. The entanglement entropy $S = -\sum s_k^2\ln s_k^2$ is 0 for a product state and $\ln 2$ for a Bell state, whose reduced states are maximally mixed ($\rho = I/2$, Bloch vector at the centre). The concurrence $C = 2\lvert\det\psi\rvert$ runs from 0 (product) to 1 (maximally entangled). Under the Ising Hamiltonian the coupling $J$ creates and destroys entanglement while the transverse field $h$ rotates the individual qubits; with $h = 0$ the Bell state is an eigenstate and nothing moves.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/quantum/two_qubit_entanglement/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import entanglement_entropy, reduced_density_1, schmidt


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    sigma = tensors.pauli(dev)

    bell = solve.bell_circuit(dev)
    _, s, _ = schmidt(bell)
    print(f"Bell state psi_ab = {bell.real.flatten().tolist()}")
    print(f"Schmidt coefficients {s.tolist()}   entropy {entanglement_entropy(bell):.6f}  (ln 2 = 0.693147)")
    print(f"reduced state of qubit 1:\n{reduced_density_1(bell).real}")

    psi0 = bell if params["initial"] == "bell" else tensors.basis_state(0, 0, dev)
    H = tensors.ising_hamiltonian(params["J"], params["h"], sigma)
    result = solve.solve(psi0, H, sigma, params["t_max"])
    S_max, C_max = result["entropy"].max(), result["concurrence"].max()
    print(f"during the evolution: max entropy {S_max:.4f}   max concurrence {C_max:.4f}")
    return plot.animate(result, out_dir / "two_qubit_entanglement.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/quantum/two_qubit_entanglement/equation.py
r"""Physics → tensor translation.

A two-qubit state |\psi\rangle = \psi_{ab} |a\rangle |b\rangle is a 2×2 tensor (a: qubit 1, b: qubit 2).

Gates. A two-qubit gate is a rank-4 tensor G_{abcd} (out: a b, in: c d); a single-qubit gate A on qubit 1
is the tensor product (A \otimes I)_{abcd} = A_{ac} \delta_{bd}. Applying a gate sums over the inputs:

    \psi_{ab} \leftarrow G_{abcd} \psi_{cd}                        (c, d summed)

Schmidt decomposition = SVD of the matrix \psi_{ab}:

    \psi_{ab} = U_{ak} s_k V^*_{bk},      S = -s_k^2 \ln s_k^2      (k summed)

Reduced density matrices (partial traces) and Bloch vectors:

    \rho^{(1)}_{ac} = \psi_{ab} \psi^*_{cb}     (b summed: trace out qubit 2)
    \rho^{(2)}_{bd} = \psi_{ab} \psi^*_{ad}     (a summed: trace out qubit 1)
    r_k = \mathrm{Tr}(\rho \sigma_k) = \sigma_{kca} \rho_{ac}

Time evolution with H_{abcd} = J Z_{ac} Z_{bd} + h (X_{ac} \delta_{bd} + \delta_{ac} X_{bd}): reshaped to a 4×4
matrix, H = W \mathrm{diag}(E) W^\dagger and U_{ij}(t) = W_{ik} e^{-i E_k t} W^*_{jk}.
"""

import torch


def apply_gate(G: torch.Tensor, psi: torch.Tensor) -> torch.Tensor:
    r"""\psi_{ab} \leftarrow G_{abcd} \psi_{cd}.   G: (2, 2, 2, 2), psi: (2, 2) → (2, 2)"""
    return torch.einsum("abcd,cd->ab", G, psi)


def schmidt(psi: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    r"""\psi_{ab} = U_{ak} s_k V^*_{bk}.   psi: (..., 2, 2) → U (..., 2, 2), s (..., 2), Vh (..., 2, 2)"""
    return torch.linalg.svd(psi)


def entanglement_entropy(psi: torch.Tensor) -> torch.Tensor:
    r"""S = -s_k^2 \ln s_k^2 (k summed), in nats.   psi: (..., 2, 2) → (...)"""
    _, s, _ = schmidt(psi)
    p = s**2  # Schmidt probabilities, sum to 1
    return -(p * torch.log(p.clamp_min(1e-30))).sum(-1)


def concurrence(psi: torch.Tensor) -> torch.Tensor:
    r"""C = 2 |\psi_{00}\psi_{11} - \psi_{01}\psi_{10}| = 2|\det\psi|.   (..., 2, 2) → (...)"""
    return 2 * torch.linalg.det(psi).abs()


def reduced_density_1(psi: torch.Tensor) -> torch.Tensor:
    r"""\rho^{(1)}_{ac} = \psi_{ab} \psi^*_{cb}  (trace out qubit 2).   (..., 2, 2) → (..., 2, 2)"""
    return torch.einsum("...ab,...cb->...ac", psi, psi.conj())


def reduced_density_2(psi: torch.Tensor) -> torch.Tensor:
    r"""\rho^{(2)}_{bd} = \psi_{ab} \psi^*_{ad}  (trace out qubit 1).   (..., 2, 2) → (..., 2, 2)"""
    return torch.einsum("...ab,...ad->...bd", psi, psi.conj())


def bloch_vector(rho: torch.Tensor, sigma: torch.Tensor) -> torch.Tensor:
    r"""r_k = \sigma_{kca} \rho_{ac} = Tr(\rho \sigma_k).   rho: (..., 2, 2), sigma: (3, 2, 2) → (..., 3)"""
    return torch.einsum("kca,...ac->...k", sigma, rho).real


def propagators(H: torch.Tensor, times: torch.Tensor) -> torch.Tensor:
    r"""U_{ij}(t) = W_{ik} e^{-i E_k t} W^*_{jk}.   H: (2, 2, 2, 2), times: (T,) → (T, 2, 2, 2, 2)"""
    E, W = torch.linalg.eigh(H.reshape(4, 4))  # pair (a, b) → row 2a + b
    phases = torch.exp(-1j * torch.einsum("t,k->tk", times.to(E.dtype), E))  # (T, 4)
    U = torch.einsum("ik,tk,jk->tij", W, phases, W.conj())  # (T, 4, 4)
    return U.reshape(-1, 2, 2, 2, 2)


def evolve(U: torch.Tensor, psi0: torch.Tensor) -> torch.Tensor:
    r"""\psi_{tab} = U_{tabcd} \psi_{cd}(0).   U: (T, 2, 2, 2, 2), psi0: (2, 2) → (T, 2, 2)"""
    return torch.einsum("tabcd,cd->tab", U, psi0)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/quantum/two_qubit_entanglement/tensors.py
"""Build the concrete tensors: Pauli matrices, gate tensors, basis states and the Ising Hamiltonian."""

import math

import torch

C = torch.complex128


def pauli(device: torch.device) -> torch.Tensor:
    """σ_{kab}: the three Pauli matrices stacked along k ∈ {x, y, z}. Shape (3, 2, 2) complex."""
    sx = [[0, 1], [1, 0]]
    sy = [[0, -1j], [1j, 0]]
    sz = [[1, 0], [0, -1]]
    return torch.tensor([sx, sy, sz], dtype=C, device=device)


def hadamard(device: torch.device) -> torch.Tensor:
    """Single-qubit Hadamard H = (X + Z)/√2. Shape (2, 2)."""
    return torch.tensor([[1, 1], [1, -1]], dtype=C, device=device) / math.sqrt(2)


def on_qubit_1(A: torch.Tensor) -> torch.Tensor:
    """(A ⊗ I)_{abcd} = A_{ac} δ_{bd}: act with A on qubit 1, leave qubit 2 alone. Shape (2, 2, 2, 2)."""
    return torch.einsum("ac,bd->abcd", A, torch.eye(2, dtype=A.dtype, device=A.device))


def on_qubit_2(B: torch.Tensor) -> torch.Tensor:
    """(I ⊗ B)_{abcd} = δ_{ac} B_{bd}. Shape (2, 2, 2, 2)."""
    return torch.einsum("ac,bd->abcd", torch.eye(2, dtype=B.dtype, device=B.device), B)


def cnot(device: torch.device) -> torch.Tensor:
    """CNOT with qubit 1 as control: G_{abcd} = 1 if a = c and b = c XOR d. Shape (2, 2, 2, 2)."""
    G = torch.zeros(2, 2, 2, 2, dtype=C, device=device)
    for c in range(2):
        for d in range(2):
            G[c, c ^ d, c, d] = 1.0  # |c, d⟩ → |c, c ⊕ d⟩
    return G


def basis_state(a: int, b: int, device: torch.device) -> torch.Tensor:
    """|a b⟩ as a tensor: ψ_{a'b'} = δ_{aa'} δ_{bb'}. Shape (2, 2)."""
    psi = torch.zeros(2, 2, dtype=C, device=device)
    psi[a, b] = 1.0
    return psi


def ising_hamiltonian(J: float, h: float, sigma: torch.Tensor) -> torch.Tensor:
    """H_{abcd} = J Z_{ac} Z_{bd} + h (X_{ac} δ_{bd} + δ_{ac} X_{bd}). Shape (2, 2, 2, 2).

    Reshaped to (4, 4) this equals torch.kron(Z, Z) * J + h * (torch.kron(X, I) + torch.kron(I, X)).
    """
    X, Z = sigma[0], sigma[2]
    zz = torch.einsum("ac,bd->abcd", Z, Z)  # Z ⊗ Z
    return J * zz + h * (on_qubit_1(X) + on_qubit_2(X))


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/quantum/two_qubit_entanglement/solve.py
"""Run the Bell circuit, then evolve the pair under the Ising Hamiltonian and measure entanglement."""

import torch

from . import tensors
from .equation import (
    apply_gate,
    bloch_vector,
    concurrence,
    entanglement_entropy,
    evolve,
    propagators,
    reduced_density_1,
    reduced_density_2,
)


def bell_circuit(device: torch.device) -> torch.Tensor:
    """|00⟩ → (H ⊗ I) → CNOT → (|00⟩ + |11⟩)/√2. Returns ψ_ab, shape (2, 2)."""
    psi = tensors.basis_state(0, 0, device)
    psi = apply_gate(tensors.on_qubit_1(tensors.hadamard(device)), psi)  # (|0⟩ + |1⟩)|0⟩ / √2
    return apply_gate(tensors.cnot(device), psi)  # flip qubit 2 when qubit 1 is |1⟩


def solve(psi0: torch.Tensor, H: torch.Tensor, sigma: torch.Tensor, t_max: float, n_steps: int = 240) -> dict:
    """Exact evolution at T = n_steps + 1 times; every quantity is computed for all times in one batch.

    Returns a dict of tensors: times (T,), psi (T, 2, 2), entropy (T,), concurrence (T,), r1 and r2 (T, 3).
    """
    times = torch.linspace(0.0, t_max, n_steps + 1, dtype=torch.float64, device=psi0.device)  # (T,)
    psi = evolve(propagators(H, times), psi0)  # (T, 2, 2)
    return {
        "times": times,
        "psi": psi,
        "entropy": entanglement_entropy(psi),
        "concurrence": concurrence(psi),
        "r1": bloch_vector(reduced_density_1(psi), sigma),
        "r2": bloch_vector(reduced_density_2(psi), sigma),
    }


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/quantum/two_qubit_entanglement/plot.py
"""Animate the two reduced Bloch vectors (left) and entanglement entropy / concurrence vs time (right)."""

import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import ACCENT, frame_indices, save_gif


def draw_sphere(ax, title: str) -> None:
    """Faint unit sphere with equator, a meridian and the x, y, z axes."""
    u = np.linspace(0, 2 * np.pi, 60)
    ax.plot(np.cos(u), np.sin(u), 0 * u, color="#9ca3af", lw=0.7)  # equator
    ax.plot(np.cos(u), 0 * u, np.sin(u), color="#d1d5db", lw=0.7)  # xz meridian
    ax.plot(0 * u, np.cos(u), np.sin(u), color="#d1d5db", lw=0.7)  # yz meridian
    for axis, name in zip(np.eye(3), "xyz"):
        ax.plot(*np.stack([-axis, axis], axis=1), color="#9ca3af", lw=0.6, ls=":")
        ax.text(*(1.25 * axis), name, color="#4b5563", ha="center", va="center", fontsize=8)
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.set_zlim(-1, 1)
    ax.set_box_aspect((1, 1, 1))
    ax.set_axis_off()
    ax.view_init(elev=20, azim=35)
    ax.set_title(title, fontsize=9)


def animate(result: dict, path: Path) -> Path:
    t = result["times"].cpu().numpy()
    S = result["entropy"].cpu().numpy() / math.log(2)  # in units of ln 2: 1 = maximally entangled
    C = result["concurrence"].cpu().numpy()
    bloch = [result["r1"].cpu().numpy(), result["r2"].cpu().numpy()]  # two × (T, 3)

    fig = plt.figure(figsize=(7, 4))
    grid = fig.add_gridspec(2, 2, width_ratios=[1, 1.5])
    arrows, tips, trails = [], [], []
    for row, colour in enumerate([ACCENT[0], ACCENT[3]]):
        ax3 = fig.add_subplot(grid[row, 0], projection="3d")
        draw_sphere(ax3, f"qubit {row + 1}: Bloch vector of ρ{row + 1}")
        trails.append(ax3.plot([], [], [], color=colour, lw=0.8, alpha=0.5)[0])
        arrows.append(ax3.plot([], [], [], color=colour, lw=2.5)[0])
        tips.append(ax3.plot([], [], [], "o", color=colour, ms=4)[0])

    ax = fig.add_subplot(grid[:, 1])
    ax.set_xlim(t[0], t[-1])
    ax.set_ylim(-0.05, 1.1)
    ax.set_xlabel("t")
    ax.set_ylabel("entanglement")
    (line_S,) = ax.plot([], [], color=ACCENT[1], lw=2, label="entropy S / ln 2")
    (line_C,) = ax.plot([], [], color=ACCENT[2], lw=1.5, ls="--", label="concurrence C")
    (line_r,) = ax.plot([], [], color=ACCENT[0], lw=1, alpha=0.7, label="|r₁| (Bloch vector length)")
    ax.legend(loc="lower right", fontsize=8)
    clock = ax.set_title("t = 0")
    r1_len = np.linalg.norm(bloch[0], axis=1)

    def update(step: int):
        for r, arrow, tip, trail in zip(bloch, arrows, tips, trails):
            trail.set_data_3d(r[: step + 1, 0], r[: step + 1, 1], r[: step + 1, 2])
            arrow.set_data_3d([0, r[step, 0]], [0, r[step, 1]], [0, r[step, 2]])
            tip.set_data_3d([r[step, 0]], [r[step, 1]], [r[step, 2]])
        line_S.set_data(t[: step + 1], S[: step + 1])
        line_C.set_data(t[: step + 1], C[: step + 1])
        line_r.set_data(t[: step + 1], r1_len[: step + 1])
        clock.set_text(f"t = {t[step]:.2f}   S = {S[step] * math.log(2):.3f}   C = {C[step]:.3f}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 90), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'initial': 'bell',
    'J': 1.0,
    'h': 0.6,
    't_max': 10.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.quantum.two_qubit_entanglement")]:
    del sys.modules[name]
main = importlib.import_module("problems.quantum.two_qubit_entanglement.main")
problem_dir = Path("problems/quantum/two_qubit_entanglement")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
